# Underwater Segmentation: Kaggle Experiments

Этот notebook предназначен для запуска экспериментов в Kaggle: проверка данных, обучение U-Net baseline, обучение DeepLabV3 ResNet50 baseline, сохранение чекпоинтов, метрик, визуализаций и архива с результатами.

## Где менять путь к датасету

Канонический Kaggle путь для SUIM dataset:

```python
DATA_DIR = /kaggle/input/datasets/ashish2001/semantic-segmentation-of-underwater-imagery-suim
```

Если dataset подключён под другим именем, поменяйте `DATASET_NAME` или задайте `DATA_DIR` напрямую в ячейке **Configuration**. Код сначала ищет `train/images` и `train/masks`, а если их нет — использует реальную структуру SUIM `train_val/images` и `train_val/masks`. Для теста используется `test/images`, иначе `TEST/images`.

Notebook не использует локальные приватные пути и не требует GPU: если CUDA недоступна, код запустится на CPU, но обучение будет медленнее.

## 1. Clone or Use Repository Code

Если notebook запущен из репозитория и рядом есть папка `src`, используется текущий код. Если нет, репозиторий клонируется в `/kaggle/working/Underwater_segmentation`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("REPO_URL", "https://github.com/MataNerdy/Underwater_segmentation.git")
REPO_DIR = Path("/kaggle/working/Underwater_segmentation")

if Path("src").exists():
    PROJECT_DIR = Path.cwd()
elif REPO_DIR.exists():
    PROJECT_DIR = REPO_DIR
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    PROJECT_DIR = REPO_DIR

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project directory: {PROJECT_DIR}")

## 2. Install Requirements If Needed

In [ ]:
required_imports = ["torch", "torchvision", "numpy", "PIL", "matplotlib", "tqdm"]
missing = []

for package_name in required_imports:
    try:
        __import__(package_name)
    except ImportError:
        missing.append(package_name)

if missing:
    print("Missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    print("All required packages are already available.")

## 3. Configuration

По умолчанию используется Kaggle dataset `datasets/ashish2001/semantic-segmentation-of-underwater-imagery-suim`. Если в вашем Kaggle notebook путь отличается, измените `DATASET_NAME` или задайте `DATA_DIR` напрямую.

Реальная структура данных:

```text
/kaggle/input/datasets/ashish2001/semantic-segmentation-of-underwater-imagery-suim/
├── train_val/images
├── train_val/masks
└── TEST/images
```

In [ ]:
from pathlib import Path
import torch

DATASET_NAME = os.environ.get("DATASET_NAME", "datasets/ashish2001/semantic-segmentation-of-underwater-imagery-suim")
DATA_DIR = Path(os.environ.get("DATA_DIR", f"/kaggle/input/{DATASET_NAME}"))

def select_existing_dir(label, *candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            print(f"Selected {label}: {candidate}")
            return candidate
    fallback = Path(candidates[-1])
    print(f"Selected {label}: {fallback} (not found yet)")
    return fallback

TRAIN_IMAGES_DIR = select_existing_dir("train images dir", DATA_DIR / "train" / "images", DATA_DIR / "train_val" / "images")
TRAIN_MASKS_DIR = select_existing_dir("train masks dir", DATA_DIR / "train" / "masks", DATA_DIR / "train_val" / "masks")
TEST_IMAGES_DIR = select_existing_dir("test images dir", DATA_DIR / "test" / "images", DATA_DIR / "TEST" / "images")

WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
EXPERIMENTS_DIR = WORKING_DIR / "experiments"
ASSETS_DIR = WORKING_DIR / "assets"

for directory in [CHECKPOINT_DIR, EXPERIMENTS_DIR, ASSETS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DATA_DIR: {DATA_DIR}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. Training will run on CPU.")

## 4. Sanity Checks: Files, Image and Mask

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

from src.dataset import IMAGE_EXTENSIONS, rgb_mask_to_index

for path in [TRAIN_IMAGES_DIR, TRAIN_MASKS_DIR, TEST_IMAGES_DIR]:
    print(path, "exists=", path.exists())
    if path.exists():
        files = sorted(p for p in path.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
        print("  files:", len(files))
        print("  first:", [p.name for p in files[:5]])

train_images = sorted(p for p in TRAIN_IMAGES_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
train_masks = sorted(p for p in TRAIN_MASKS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
assert train_images, f"No train images found in {TRAIN_IMAGES_DIR}"
assert train_masks, f"No train masks found in {TRAIN_MASKS_DIR}"

image_path = train_images[0]
mask_candidates = [p for p in train_masks if p.stem == image_path.stem]
assert mask_candidates, f"No matching mask found for {image_path.name}"
mask_path = mask_candidates[0]

image = Image.open(image_path).convert("RGB")
mask_rgb = Image.open(mask_path).convert("RGB")
mask_idx = rgb_mask_to_index(mask_rgb)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image)
axes[0].set_title(f"Image: {image_path.name}")
axes[1].imshow(mask_rgb)
axes[1].set_title(f"RGB mask: {mask_path.name}")
axes[2].imshow(mask_idx, vmin=0, vmax=7, cmap="tab10")
axes[2].set_title(f"Encoded classes: {np.unique(mask_idx).tolist()}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 5. Experiment Settings

Для быстрого smoke run можно оставить `EPOCHS = 1`. Для полноценного сравнения увеличьте `EPOCHS`, например до `10` или `20`.

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", "1"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "8"))
IMAGE_SIZE = int(os.environ.get("IMAGE_SIZE", "256"))
LR = float(os.environ.get("LR", "1e-4"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "2"))

print({
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "lr": LR,
    "num_workers": NUM_WORKERS,
})

## 6. Train U-Net Baseline

Чекпоинт будет сохранён в `/kaggle/working/checkpoints/unet/best.pth`, результаты — в `/kaggle/working/experiments/results.csv`.

In [ ]:
unet_checkpoint_dir = CHECKPOINT_DIR / "unet"
unet_checkpoint_dir.mkdir(parents=True, exist_ok=True)

!python src/train.py \
  --model unet \
  --images-dir "{TRAIN_IMAGES_DIR}" \
  --masks-dir "{TRAIN_MASKS_DIR}" \
  --checkpoint-dir "{unet_checkpoint_dir}" \
  --results-csv "{EXPERIMENTS_DIR / 'results.csv'}" \
  --assets-dir "{ASSETS_DIR}" \
  --experiment-name unet_kaggle_baseline \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --image-size {IMAGE_SIZE} \
  --lr {LR} \
  --num-workers {NUM_WORKERS} \
  --device {DEVICE}

UNET_CHECKPOINT = unet_checkpoint_dir / "best.pth"
print("U-Net checkpoint:", UNET_CHECKPOINT)

## 7. Train DeepLabV3 ResNet50 Baseline

По умолчанию `USE_PRETRAINED_BACKBONE = False`, чтобы notebook не требовал скачивания весов. Если интернет в Kaggle включён и нужны pretrained weights, поставьте `True`.

In [ ]:
USE_PRETRAINED_BACKBONE = os.environ.get("USE_PRETRAINED_BACKBONE", "0") == "1"
deeplab_checkpoint_dir = CHECKPOINT_DIR / "deeplabv3_resnet50"
deeplab_checkpoint_dir.mkdir(parents=True, exist_ok=True)
pretrained_flag = "--pretrained-backbone" if USE_PRETRAINED_BACKBONE else ""

!python src/train.py \
  --model deeplabv3_resnet50 \
  {pretrained_flag} \
  --images-dir "{TRAIN_IMAGES_DIR}" \
  --masks-dir "{TRAIN_MASKS_DIR}" \
  --checkpoint-dir "{deeplab_checkpoint_dir}" \
  --results-csv "{EXPERIMENTS_DIR / 'results.csv'}" \
  --assets-dir "{ASSETS_DIR}" \
  --experiment-name deeplabv3_resnet50_kaggle_baseline \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --image-size {IMAGE_SIZE} \
  --lr {LR} \
  --num-workers {NUM_WORKERS} \
  --device {DEVICE}

DEEPLAB_CHECKPOINT = deeplab_checkpoint_dir / "best.pth"
print("DeepLabV3 checkpoint:", DEEPLAB_CHECKPOINT)

## 8. Inspect Experiment Results

In [ ]:
results_path = EXPERIMENTS_DIR / "results.csv"

try:
    import pandas as pd
    display(pd.read_csv(results_path))
except ImportError:
    print(results_path.read_text())

## 9. Save Prediction Visualization Examples

Визуализация ниже использует U-Net checkpoint. При желании замените `UNET_CHECKPOINT` на `DEEPLAB_CHECKPOINT`.

In [ ]:
!python src/visualize.py \
  --images-dir "{TRAIN_IMAGES_DIR}" \
  --masks-dir "{TRAIN_MASKS_DIR}" \
  --checkpoint "{UNET_CHECKPOINT}" \
  --output "{ASSETS_DIR / 'prediction_examples.png'}" \
  --num-examples 6 \
  --device {DEVICE}

from IPython.display import Image as IPyImage, display
prediction_examples = ASSETS_DIR / "prediction_examples.png"
display(IPyImage(filename=str(prediction_examples)))

## 10. Optional: Generate Test Submission with Horizontal Flip TTA

Эта ячейка создаёт `/kaggle/working/submission.txt`, если есть `test/images`.

In [ ]:
if TEST_IMAGES_DIR.exists():
    !python src/predict.py \
      --images-dir "{TEST_IMAGES_DIR}" \
      --checkpoint "{UNET_CHECKPOINT}" \
      --output "{WORKING_DIR / 'submission.txt'}" \
      --predictions-dir "{WORKING_DIR / 'predictions'}" \
      --tta \
      --device {DEVICE}
else:
    print(f"Skipping prediction: {TEST_IMAGES_DIR} does not exist")

## 11. Zip Outputs

Архив будет сохранён как `/kaggle/working/underwater_experiments_outputs.zip`.

In [ ]:
import zipfile

zip_path = WORKING_DIR / "underwater_experiments_outputs.zip"
paths_to_zip = [CHECKPOINT_DIR, EXPERIMENTS_DIR, ASSETS_DIR, WORKING_DIR / "submission.txt", WORKING_DIR / "predictions"]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths_to_zip:
        if not path.exists():
            continue
        if path.is_file():
            archive.write(path, arcname=path.relative_to(WORKING_DIR))
        else:
            for file_path in path.rglob("*"):
                if file_path.is_file():
                    archive.write(file_path, arcname=file_path.relative_to(WORKING_DIR))

print(f"Saved: {zip_path}")